# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the dataset description
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}\n")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The FAIR² dataset may contain multiple record sets. Each record set groups together related records (such as tabular data or annotation sets). We'll list out all record sets and their key fields by their `@id`.

In [ ]:
# List all record sets and their fields
# Entities are referenced by their '@id'

def show_record_sets(dataset):
    print('== Available Record Sets ==\n')
    record_sets = dataset.metadata.recordSet
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        print(f"- Record Set @id: {rs_id}")
        print(f"  Name: {getattr(rs, 'name', '')}")
        print(f"  Description: {getattr(rs, 'description', '')}")
        # List fields and columns
        if hasattr(rs, 'field') and rs.field:
            print("  Fields:")
            for field in rs.field:
                print(f"    - Field @id: {getattr(field, '@id', '')}")
                print(f"      Name: {getattr(field, 'name', '')}")
                print(f"      Data Type: {getattr(field, 'dataType', '')}")
                if hasattr(field, 'column') and field.column:
                    for col in field.column:
                        print(f"      Column @id: {getattr(col, '@id', '')}")
        print()

show_record_sets(dataset)

## 2B. Preview Records from a Record Set

Now, let's quickly preview sample records from one of the record sets. Use the correct `@id` as shown above.

In [ ]:
# Example: Print several sample records from a chosen record set

# Fetch all record set IDs
record_sets = dataset.metadata.recordSet
all_rs_ids = [getattr(rs, '@id') for rs in record_sets]

# Choose the first available record set for demonstration:
if all_rs_ids:
    selected_rs_id = all_rs_ids[0]
    print(f"Previewing records from record set '@id': {selected_rs_id}\n")
    for i, record in enumerate(dataset.records(record_set=selected_rs_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis.

Entities must be referenced by record set and field `@id`.

In [ ]:
# For each available record set, load its records into a DataFrame
dfs = {}
for rs_id in all_rs_ids:
    print(f"Loading records from record set: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if len(recs) > 0:
        dfs[rs_id] = pd.DataFrame(recs)
        print(f"  --> Columns: {list(dfs[rs_id].columns)}")
        print(dfs[rs_id].head(2))
    else:
        print("  [No records found]")
    print("-")

# For EDA, we'll select the primary record set (the first one)
if all_rs_ids:
    main_rs_id = all_rs_ids[0]
    print(f"Selected main record set: {main_rs_id}")
    main_df = dfs[main_rs_id]
    print("\nAvailable columns (field @id):")
    print(main_df.columns.tolist())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
We demonstrate common data processing steps such as filtering, normalization, and grouping. All columns/fields are referenced by their `@id`.

**Note:** Adjust `numeric_field_id` and `group_field_id` according to available field/column `@id`s shown above.

In [ ]:
if main_rs_id:
    df = main_df.copy()
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    # Choose the first numeric field as example
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        # Allow the user to edit this, e.g. the field with MW, Coverage, etc.
        numeric_field_id = df.columns[0]  # fallback

    print(f"Analyzing numeric field: {numeric_field_id}\n")
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean):")
    print(filtered_df.head(), '\n')

    # Normalize the field
    normalized_col = numeric_field_id + "_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' values:")
    print(filtered_df[[numeric_field_id, normalized_col]].head(), '\n')

    # Attempt to group by a categorical/text column if available
    categorical_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
    group_field_id = None
    for c in categorical_candidates:
        if c != numeric_field_id:
            group_field_id = c
            break

    if group_field_id:
        print(f"Grouping by field: '{group_field_id}'\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No appropriate group field found for grouping.\n")
else:
    print("No main record set loaded; skipping EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

This uses matplotlib and seaborn for simple data visualizations such as histograms and scatterplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id exists, boxplot by group
    if group_field_id:
        plt.figure(figsize=(10, 4))
        # Limit to top 10 categories by count for clarity
        top_cats = df[group_field_id].value_counts().index[:10]
        plot_df = df[df[group_field_id].isin(top_cats)]
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric or group fields found for visualization.")

## 6. Conclusion
In this notebook, we've loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset via its Croissant schema, examined its record sets and fields by their `@id`, extracted record set(s) into DataFrames, and performed basic EDA and visualizations. All references to data elements are made explicitly by their Croissant `@id` as per best practices.

- Use the overview and DataFrame column listing above to identify field/column `@id` values appropriate for further deep dives.
- The workflow shown enables advanced analysis, filtering, and visualization on provenance-tracked, FAIR-compliant datasets using `mlcroissant`.

For more advanced transform and integration operations, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/).
